# Stable Audio 3 — LANDR Evaluation Notebook

| Section | Capability | Prompts |
|---------|-----------|--------|
| 1 | **Generate from scratch** | Curated in config.yaml |
| 2 | **Variation** — all 23 samples × 3 noise levels | Diverse pool, unrelated to originals |
| 3 | **Extend** — all 23 samples → 2× duration | Same diverse pool |

Edit `config.yaml` to change model, steps, seed, or prompts.

## Section 0 — Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import yaml
import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from IPython.display import display, Audio, HTML

from stable_audio_3 import StableAudioModel
print('Imports OK')


In [ ]:
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

SAMPLES_DIR     = Path(cfg['samples_dir'])
RUN_DIR         = Path(cfg['output_dir']) / datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR.mkdir(parents=True, exist_ok=True)

SEED            = cfg['seed']            # -1 = random — used in sections 2 & 3
COMPARISON_SEED = cfg['comparison_seed'] # fixed — used in section 1 step comparison only
NOISE_LEVELS    = cfg['variation_noise_levels']
MAX_DUR         = cfg['max_duration']
PROMPT_POOL     = cfg['prompt_pool']

samples = sorted(SAMPLES_DIR.glob('*.wav'))
print(f'Output dir      : {RUN_DIR}')
print(f'Model           : {cfg["model"]}')
print(f'Samples         : {len(samples)} .wav files')
print(f'Prompt pool     : {len(PROMPT_POOL)} prompts')
print(f'Seed (explore)  : {SEED}  ← random, sections 2 & 3')
print(f'Seed (compare)  : {COMPARISON_SEED}  ← fixed, section 1 step comparison')

In [ ]:
model = StableAudioModel.from_pretrained(cfg['model'])
SR    = model.model_config['sample_rate']
print(f'Model loaded — sample rate: {SR} Hz')


In [ ]:
def save_wav(audio, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    wav = audio[0].cpu() if audio.dim() == 3 else audio.cpu()
    torchaudio.save(str(path), wav, SR)
    return path

def play(path, label=''):
    display(HTML(f'<b style="font-family:monospace;font-size:12px">{label}</b>' if label else ''))
    display(Audio(str(path)))

def plot_waveforms(paths_labels, title=''):
    n = len(paths_labels)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 2), sharey=True)
    if n == 1: axes = [axes]
    for ax, (p, lbl) in zip(axes, paths_labels):
        wav, sr = torchaudio.load(str(p))
        mono = wav.mean(0).numpy()
        t = np.linspace(0, len(mono)/sr, len(mono))
        ax.plot(t, mono, linewidth=0.4)
        ax.set_title(lbl, fontsize=8)
        ax.set_xlabel('s')
        ax.set_ylim(-1.05, 1.05)
    if title: fig.suptitle(title, fontsize=10, y=1.02)
    plt.tight_layout()
    plt.show()

def load_audio(path):
    wav, sr = torchaudio.load(str(path))
    return sr, wav

def audio_duration(path):
    info = torchaudio.info(str(path))
    return info.num_frames / info.sample_rate

def section_header(title):
    display(HTML(f'<h3 style="border-left:4px solid #555;padding-left:8px;margin-top:24px">{title}</h3>'))

print('Helpers ready.')


---
## Section 1 — Generate from Scratch

Pure text-to-audio. No audio input — model generates entirely from the prompt.

Each prompt is generated at **4 step levels (8 / 30 / 40 / 50)** for quality comparison.

> **Why do the same prompt sound similar across step levels?**  
> All 4 levels use `comparison_seed=42` — a fixed seed means they all start from identical noise and converge to the same audio character. You are hearing **quality differences** (detail, coherence), not random variation.  
> To get completely different outputs, change `comparison_seed` in `config.yaml`.

The notebook includes both **technical prompts** (producer-style with BPM/key) and **human prompts** (natural language) so you can compare how the model interprets each style.


In [ ]:
gen_dir = RUN_DIR / '1_generate'
gen_dir.mkdir(exist_ok=True)

STEPS_LEVELS = cfg['generate_steps_levels']  # [8, 30, 40, 50]

for item in cfg['generate_prompts']:
    label  = item['label']
    prompt = item['prompt']
    dur    = item['duration']

    print(f'\n▶ {label} | {dur}s')
    print(f'  {prompt}')

    outputs = []
    for steps in STEPS_LEVELS:
        out = gen_dir / f'{label}_steps{steps}.wav'
        print(f'  steps={steps}...', end=' ', flush=True)
        audio = model.generate(
            prompt=prompt, duration=dur,
            steps=steps, cfg_scale=cfg['generate_cfg'],
            seed=COMPARISON_SEED,  # fixed seed — isolates steps as the only variable
        )
        save_wav(audio, out)
        outputs.append((out, f'steps={steps}'))
        print('done')

    plot_waveforms(outputs, title=f'{label} — step comparison')
    for path, lbl in outputs:
        play(path, label=f'{label} | {lbl} | "{prompt}"')

print('\n✅ Section 1 complete.')


---
## Section 2 — Variation (Audio-to-Audio)

> **A prompt is always required** by the API — `prompt=None` raises an `AssertionError`.
> To approximate a promptless variation, we use a neutral prompt (`"audio"`) with `cfg_scale=1.0` (minimal guidance).

All 23 LANDR samples × 3 noise levels, tested with **neutral prompt** vs **creative prompt**.

| | Neutral prompt `"audio"` | Creative prompt (diverse pool) |
|--|--|--|
| **Direction** | None — variation driven purely by noise level | Style/genre steered by the prompt |
| **Noise 0.3** | Very close to the original | Slight tint toward the prompt style |
| **Noise 0.9** | Free variation, no stylistic direction | Strongly pushed toward the prompt |

Each sample: `original → neutral (0.3 / 0.6 / 0.9) → creative (0.3 / 0.6 / 0.9)`

In [ ]:
var_dir = RUN_DIR / '2_variation'
var_dir.mkdir(exist_ok=True)

for i, sample_path in enumerate(samples):
    prompt   = PROMPT_POOL[i % len(PROMPT_POOL)]
    dur      = min(audio_duration(sample_path), MAX_DUR)
    src_sr, src_wav = load_audio(sample_path)
    stem     = sample_path.stem
    sdir     = var_dir / stem
    sdir.mkdir(exist_ok=True)

    out_orig = sdir / 'original.wav'
    torchaudio.save(str(out_orig), src_wav, src_sr)

    section_header(f'Variation {i+1}/23 — {sample_path.name}')
    print(f'  Creative prompt : {prompt}')
    print(f'  Duration: {dur:.1f}s | steps={cfg["variation_steps"]}')

    # --- Neutral prompt (closest approximation to "promptless") ---
    neutral_outputs = [(out_orig, 'original')]
    for noise in NOISE_LEVELS:
        out = sdir / f'neutral_noise_{noise}.wav'
        audio = model.generate(
            prompt='audio', duration=dur,
            steps=cfg['variation_steps'], cfg_scale=1.0, seed=SEED,
            init_audio=(src_sr, src_wav), init_noise_level=noise,
        )
        save_wav(audio, out)
        neutral_outputs.append((out, f'neutral | noise {noise}'))
        print(f'  [neutral] noise={noise} done')

    # --- Creative prompt ---
    creative_outputs = [(out_orig, 'original')]
    for noise in NOISE_LEVELS:
        out = sdir / f'creative_noise_{noise}.wav'
        audio = model.generate(
            prompt=prompt, duration=dur,
            steps=cfg['variation_steps'], cfg_scale=cfg['variation_cfg'], seed=SEED,
            init_audio=(src_sr, src_wav), init_noise_level=noise,
        )
        save_wav(audio, out)
        creative_outputs.append((out, f'creative | noise {noise}'))
        print(f'  [creative] noise={noise} done')

    display(HTML('<b>Neutral prompt — "audio"</b>'))
    plot_waveforms(neutral_outputs, title=f'{stem} — neutral')
    for path, lbl in neutral_outputs:
        play(path, label=lbl)

    display(HTML(f'<b>Creative prompt — "{prompt}"</b>'))
    plot_waveforms(creative_outputs, title=f'{stem} — creative')
    for path, lbl in creative_outputs:
        play(path, label=lbl)

print('\n✅ Section 2 complete.')


---
## Section 3 — Extend (Continuation)

All 23 samples extended via inpainting across a **full parameter grid**. The original is kept intact; model generates only the new portion.

| Parameter | Values tested |
|-----------|--------------|
| **Duration** | 2× and 3× the original length (capped at 120s) |
| **Steps** | 8 (fast) and 50 (best quality) |
| **CFG scale** | 1.0 (loop-like, close to original style) and 7.0 (strongly evolved) |

→ 8 generated files per sample (2 durations × 2 steps × 2 CFG)

In [ ]:
ext_dir = RUN_DIR / '3_extend'
ext_dir.mkdir(exist_ok=True)

EXT_ADD_SECS = cfg['extension_add_seconds']   # e.g. [5, 15]
EXT_STEPS    = cfg['extension_steps_levels']  # e.g. [30, 50]
EXT_CFGS     = cfg['extension_cfg_levels']    # e.g. [1.0, 3.0, 7.0]

for i, sample_path in enumerate(samples):
    prompt   = PROMPT_POOL[i % len(PROMPT_POOL)]
    orig_dur = audio_duration(sample_path)
    src_sr, src_wav = load_audio(sample_path)
    stem     = sample_path.stem
    sdir     = ext_dir / stem
    sdir.mkdir(exist_ok=True)

    out_orig = sdir / 'original.wav'
    torchaudio.save(str(out_orig), src_wav, src_sr)

    section_header(f'Extend {i+1}/23 — {sample_path.name}')
    print(f'  Prompt  : {prompt}')
    print(f'  Original: {orig_dur:.1f}s')

    for add_sec in EXT_ADD_SECS:
        target_dur = min(orig_dur + add_sec, MAX_DUR)
        if target_dur <= orig_dur + 0.5:
            print(f'  Skip +{add_sec}s — already at max duration')
            continue
        print(f'  +{add_sec}s → {target_dur:.1f}s')
        for steps in EXT_STEPS:
            for cfg_val in EXT_CFGS:
                fname = f'add{add_sec}s_steps{steps}_cfg{cfg_val}.wav'
                out   = sdir / fname
                audio = model.generate(
                    prompt=prompt, duration=target_dur,
                    steps=steps, cfg_scale=cfg_val, seed=SEED,
                    inpaint_audio=(src_sr, src_wav),
                    inpaint_mask_start_seconds=orig_dur,
                    inpaint_mask_end_seconds=target_dur,
                )
                save_wav(audio, out)
                print(f'    +{add_sec}s steps={steps} cfg={cfg_val} done')

    # Display
    play(out_orig, label=f'original ({orig_dur:.1f}s)')
    for add_sec in EXT_ADD_SECS:
        target_dur = min(orig_dur + add_sec, MAX_DUR)
        if target_dur <= orig_dur + 0.5:
            continue
        display(HTML(f'<b style="margin-top:12px;display:block">+{add_sec}s → {target_dur:.1f}s total</b>'))
        grid_items = []
        for steps in EXT_STEPS:
            for cfg_val in EXT_CFGS:
                p = sdir / f'add{add_sec}s_steps{steps}_cfg{cfg_val}.wav'
                grid_items.append((p, f'steps={steps} cfg={cfg_val}'))
        plot_waveforms([(out_orig, 'original'), *grid_items], title=f'{stem} +{add_sec}s')
        for p, lbl in grid_items:
            play(p, label=lbl)

print('\n✅ Section 3 complete.')

---
## Section 4 — Summary

In [ ]:
all_wavs = sorted(RUN_DIR.rglob('*.wav'))
rows = []
for p in all_wavs:
    rel  = p.relative_to(RUN_DIR)
    dur  = audio_duration(p)
    rows.append({'section': rel.parts[0], 'file': str(rel), 'dur': f'{dur:.1f}s', 'size': f'{p.stat().st_size//1024} KB'})

if rows:
    header = '<tr>' + ''.join(f'<th style="padding:4px 8px;background:#eee">{k}</th>' for k in rows[0]) + '</tr>'
    body   = ''.join('<tr>' + ''.join(f'<td style="padding:3px 8px;font-size:11px">{v}</td>' for v in r.values()) + '</tr>' for r in rows)
    display(HTML(f'<h3>Run: {RUN_DIR.name} — {len(all_wavs)} files</h3><table border="1" style="border-collapse:collapse">{header}{body}</table>'))
